# Hard Case: Lazy-load / Infinite Scroll

*Infinite scroll* pages load new data as you scroll down (e.g. social media feeds,
product catalogs). There are **two approaches**:

- **Approach A (recommended): find the hidden API.** A scroll page usually calls a
  JSON endpoint behind the scenes. Open **DevTools → Network → XHR**, find the endpoint,
  then call it directly with `requests`. **Fast, lightweight, clean.**
- **Approach B: Selenium scroll.** If there's no usable API, we scroll
  with a browser until the data stops growing.

We'll practice on `https://quotes.toscrape.com/scroll` (which behind the scenes calls
`https://quotes.toscrape.com/api/quotes?page=N`).

**Tooling:** `requests` (approach A), `selenium` (approach B).


## Approach A — Hidden API (most efficient)

The `…/api/quotes?page=N` endpoint returns JSON with a `has_next` field. We loop
page by page until `has_next == False`. No browser needed at all.


In [1]:
import requests

API = "https://quotes.toscrape.com/api/quotes"


def fetch_via_api():
    all_quotes = []
    page = 1
    while True:
        r = requests.get(API, params={"page": page}, timeout=10)
        r.raise_for_status()
        data = r.json()
        for q in data["quotes"]:
            all_quotes.append(
                {"text": q["text"], "author": q["author"]["name"], "tags": q["tags"]}
            )
        print(f"page {page}: +{len(data['quotes'])}  (has_next={data['has_next']})")
        if not data["has_next"]:  # no next page -> done
            break
        page += 1
    return all_quotes


api_result = fetch_via_api()
print("\nTotal quotes (via API):", len(api_result))
api_result[0]


page 1: +10  (has_next=True)


page 2: +10  (has_next=True)


page 3: +10  (has_next=True)


page 4: +10  (has_next=True)


page 5: +10  (has_next=True)


...

page 7: +10  (has_next=True)


page 8: +10  (has_next=True)


page 9: +10  (has_next=True)


page 10: +10  (has_next=False)

Total quotes (via API): 100


{'text': '“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”',
 'author': 'Albert Einstein',
 'tags': ['change', 'deep-thoughts', 'thinking', 'world']}

## Approach B — Selenium scroll

If there's no API, we scroll the page with a browser. The pattern: scroll to the bottom →
pause so new content can load → count the elements. **Stop when the element count no
longer grows** (meaning everything has loaded).


In [2]:
import time

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By


def make_driver(headless=True):
    o = Options()
    if headless:  # set False if you want to watch the scrolling
        o.add_argument("--headless=new")
    o.add_argument("--window-size=1280,900")
    return webdriver.Chrome(options=o)


driver = make_driver()
try:
    driver.get("https://quotes.toscrape.com/scroll")
    previous_count = 0
    while True:
        # scroll to the bottom of the page
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)  # give new content time to load (lazy-load)

        quotes = driver.find_elements(By.CSS_SELECTOR, ".quote")
        if len(quotes) == previous_count:  # no increase -> everything loaded
            break
        previous_count = len(quotes)
        print("quotes after scroll:", previous_count)

    print("\nTotal quotes (via Selenium scroll):", previous_count)
finally:
    driver.quit()


quotes after scroll: 20


quotes after scroll: 30


quotes after scroll: 40


quotes after scroll: 50


quotes after scroll: 60


quotes after scroll: 70


quotes after scroll: 100



Total quotes (via Selenium scroll): 100


## Conclusion & Exercise

- **Always check the Network tab first** for a hidden API — it's usually far faster and more stable
  than scrolling with Selenium.
- Scroll-stop pattern: compare the element count before vs. after; equal → stop.
  (A more sophisticated alternative: compare `document.body.scrollHeight`.)
- The two approaches must yield **the same number of quotes**.

**Exercise:** load the API result into a `pandas.DataFrame`, then find the 5 most frequent tags.


In [3]:
# Sample solution to the exercise
import pandas as pd

df = pd.DataFrame(api_result)
print("DataFrame shape:", df.shape)

# tags is a list per row -> explode into one tag per row, then count
top_tags = df.explode("tags")["tags"].value_counts().head(5)
print("\nTop 5 tags:")
print(top_tags)


DataFrame shape: (100, 3)

Top 5 tags:
tags
love             14
inspirational    13
life             13
humor            12
books            11
Name: count, dtype: int64
